# SG Alerts — Checkpoint Vision Distillation (Auto-Save to Drive + Auto-Shutdown)

Uses only **~1–3 compute units** out of your 200 monthly Colab Pro credits and **automatically disconnects & deletes the GPU runtime** when finished so zero idle credits are wasted.

### How it works
1. **Mounts Google Drive** (`/content/drive/MyDrive/sgalerts_distill/`) so frames and your exported model are saved permanently.
2. **Collects live LTA frames** (`2701`, `2702`, `4703`, `4713`) — or uses frames already saved in your Drive folder.
3. **Auto-labels (Teacher)** on GPU using `YOLO11x + SAHI` tiled detection.
4. **Fine-tunes (Student)** a fast `YOLO11s` detector on these exact 4 Singapore checkpoint angles and exports **`sgalerts_custom.onnx`** directly to Google Drive.
5. **Auto-shuts down GPU** via `google.colab.runtime.unassign()` so you never forget to disconnect.

After downloading `sgalerts_custom.onnx` from Drive, place it at `.cache/models/sgalerts_custom.onnx` — `scripts/analyze-checkpoints.py` auto-detects the 4-class YOLO11 ONNX format and runs it on GitHub Actions CPU for **$0/month**.

In [ ]:
# 1. Mount Google Drive, check GPU, and install tools
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/sgalerts_distill")
RAW_DIR = DRIVE_ROOT / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
print("Drive workspace ready:", DRIVE_ROOT)

!nvidia-smi
!pip -q install requests pillow onnx onnxruntime ultralytics sahi

In [ ]:
# 2. Collect snapshots from LTA data.gov.sg into Google Drive
# Tip: Set ROUNDS = 10 (every 60s) to grab 40 frames in 10 mins, or run this cell
# once during morning rush, once at noon, and once at night so the model sees all lighting!
import io, time, requests
from PIL import Image

CAM_IDS = {"2701", "2702", "4703", "4713"}
ROUNDS = 5          # 5 rounds x 4 cameras = 20 fresh frames per run
INTERVAL_SEC = 60   # LTA refreshes cameras roughly every 1-3 minutes

for r_idx in range(ROUNDS):
    resp = requests.get("https://api.data.gov.sg/v1/transport/traffic-images", timeout=20)
    resp.raise_for_status()
    cams = resp.json()["items"][0]["cameras"]
    ts = int(time.time())
    saved = 0
    for cam in cams:
        cid = str(cam.get("camera_id", ""))
        if cid in CAM_IDS:
            img_bytes = requests.get(cam["image"], timeout=20).content
            img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            img.save(RAW_DIR / f"cam_{cid}_{ts}.jpg", quality=95)
            saved += 1
    print(f"Round {r_idx + 1}/{ROUNDS}: saved {saved} frames (total in Drive: {len(list(RAW_DIR.glob('*.jpg')))})")
    if r_idx < ROUNDS - 1:
        time.sleep(INTERVAL_SEC)

In [ ]:
# 3. Auto-label all frames in Drive with SAHI tiled YOLO11x Teacher + 85/15 train/val split
import random
from pathlib import Path
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

COCO_TO_SGA = {2: 0, 3: 1, 5: 2, 7: 3}  # car, motorcycle, bus, truck
DATASET_DIR = Path("/content/sgalerts_dataset")
for sub in ("images/train", "images/val", "labels/train", "labels/val"):
    (DATASET_DIR / sub).mkdir(parents=True, exist_ok=True)

raw_files = sorted(RAW_DIR.glob("*.jpg"))
assert len(raw_files) >= 12, f"Need at least 12 frames in {RAW_DIR} (found {len(raw_files)}). Run Cell 2 more times across day/night!"
random.Random(42).shuffle(raw_files)
val_cut = max(2, int(len(raw_files) * 0.15))
val_set = set(raw_files[:val_cut])

teacher = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="yolo11x.pt",
    confidence_threshold=0.22,
    device="cuda:0",
)

for img_path in raw_files:
    split = "val" if img_path in val_set else "train"
    im = Image.open(img_path).convert("RGB")
    W, H = im.size
    res = get_sliced_prediction(
        str(img_path),
        teacher,
        slice_height=512,
        slice_width=512,
        overlap_height_ratio=0.25,
        overlap_width_ratio=0.25,
    )
    lines = []
    for pred in res.object_prediction_list:
        cid = pred.category.id
        if cid not in COCO_TO_SGA:
            continue
        x1, y1, x2, y2 = pred.bbox.to_xyxy()
        cx = ((x1 + x2) / 2) / W
        cy = ((y1 + y2) / 2) / H
        bw = (x2 - x1) / W
        bh = (y2 - y1) / H
        lines.append(f"{COCO_TO_SGA[cid]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    im.save(DATASET_DIR / f"images/{split}" / img_path.name, quality=95)
    (DATASET_DIR / f"labels/{split}" / f"{img_path.stem}.txt").write_text("\n".join(lines), encoding="utf-8")
    print(f"[{split}] {img_path.name}: {len(lines)} vehicles")

In [ ]:
# 4. Fine-tune compact YOLO11s Student, save sgalerts_custom.onnx to Google Drive, and AUTO-SHUTDOWN GPU
import shutil
from pathlib import Path
from ultralytics import YOLO

AUTO_SHUTDOWN_GPU = True  # Automatically disconnects & deletes GPU runtime when done!

yaml_path = Path("/content/sgalerts_dataset/data.yaml")
yaml_path.write_text(
    "path: /content/sgalerts_dataset\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    "  0: car\n"
    "  1: motorcycle\n"
    "  2: bus\n"
    "  3: truck\n",
    encoding="utf-8",
)

try:
    student = YOLO("yolo11s.pt")
    student.train(data=str(yaml_path), epochs=30, imgsz=640, batch=16)
    onnx_out = Path(student.export(format="onnx", simplify=True, imgsz=640))
    drive_model = DRIVE_ROOT / "sgalerts_custom.onnx"
    shutil.copy2(onnx_out, drive_model)
    print(f"Saved custom ONNX model to Google Drive: {drive_model} ({drive_model.stat().st_size / 1e6:.1f} MB)")
finally:
    if AUTO_SHUTDOWN_GPU:
        print("Training complete — auto-disconnecting and deleting GPU runtime to save Colab credits...")
        from google.colab import runtime
        runtime.unassign()